# 6 — QPROP Batch Run

**Author:** Héctor Fernández Pinacho  
**Project:** IDEAL Lab — UAV Propeller Design Optimisation Pipeline

This notebook runs the aerodynamic propeller simulations with **QPROP**.

The previous pipeline stage generated one QPROP input file per propeller configuration. This notebook takes those files, executes QPROP in batch mode, parses the numerical output, computes the main performance metrics, and exports a compact CSV table.

The notebook has one responsibility only:

```text
QPROP input files  →  QPROP simulations  →  propeller performance CSV
```

It does not generate geometry, airfoils, XFOIL polars, or QPROP input files.


## 1. Simulation logic

For every propeller input file in `./qprop_input`, the notebook runs QPROP at a fixed operating point:

$$
V_\infty = 0
$$

$$
\text{rpm} = \text{constant}
$$

This corresponds to a static hover-style operating condition.

The main quantities extracted from QPROP are:

| Symbol | Meaning | Unit |
|---|---|---|
| `T` | thrust | N |
| `Q` | torque | N·m |
| `Pshaft` | shaft power | W |
| `CT` | thrust coefficient | - |
| `CP` | power coefficient | - |
| `cl_avg` | average blade lift coefficient | - |
| `cd_avg` | average blade drag coefficient | - |

The notebook also computes two derived metrics:

$$
\mathrm{FOM} = \frac{T^{3/2}}{\sqrt{2 \rho A} \; P_{shaft}}
$$

where

$$
A = \pi R^2
$$

and

$$
\eta_{aero} = \frac{C_T}{C_P^{2/3}}
$$

These metrics are only computed when the corresponding QPROP result is physically meaningful, meaning positive thrust and positive shaft power.


In [1]:

# Imports

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


c:\Users\hecto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Constants and paths

All tunable settings are defined here. In normal use, this is the only section that should need editing.

The notebook supports a short temporary runtime folder because some QPROP builds are sensitive to long Windows paths. The generated CSV is still saved in the project folder under `./csv`.


In [2]:

# -----------------------------
# Project paths
# -----------------------------

BASE_DIR = Path.cwd()
CSV_DIR = BASE_DIR / "csv"

QPROP_INPUT_DIR = BASE_DIR / "qprop_input"
QPROP_PROJECT_OUTPUT_DIR = BASE_DIR / "qprop_output"
QPROP_RESULTS_CSV_PATH = CSV_DIR / "qprop_batch_results.csv"

QPROP_INPUT_EXTENSION = ".txt"


# -----------------------------
# External executable and motor file
# -----------------------------

QPROP_EXE_PATH = BASE_DIR / "qprop" / "qprop.exe"
MOTOR_FILE_SOURCE = BASE_DIR / "motor.mas"


# -----------------------------
# Operating point
# -----------------------------

FREESTREAM_VELOCITY_M_S = 0.0
RPM = 4000


# -----------------------------
# Physical constants
# -----------------------------

RHO_AIR_KG_M3 = 1.225


# -----------------------------
# QPROP runtime path control
# -----------------------------

# QPROP can behave badly with long file paths on Windows.
# If True, input files and motor.mas are copied to a short runtime folder before execution.
USE_SHORT_RUNTIME_PATHS = True

SHORT_RUNTIME_ROOT = Path("C:/qp") if os.name == "nt" else BASE_DIR / "_qprop_runtime"
RUNTIME_PROP_DIR = SHORT_RUNTIME_ROOT / "props"
RUNTIME_OUTPUT_DIR = SHORT_RUNTIME_ROOT / "out"
RUNTIME_MOTOR_FILE = SHORT_RUNTIME_ROOT / "motor.mas"

# Copy the raw QPROP output files back to ./qprop_output after the batch finishes.
SYNC_RAW_OUTPUTS_TO_PROJECT_DIR = True


# -----------------------------
# Batch execution settings
# -----------------------------

MAX_WORKERS = min(os.cpu_count() or 4, 16)
QPROP_TIMEOUT_S = 30
OVERWRITE_EXISTING_OUTPUTS = False


# -----------------------------
# Output formatting
# -----------------------------

CONFIG_ID_DIGITS = 4

QPROP_COLUMNS = [
    "V", "rpm", "Dbeta", "T", "Q", "Pshaft", "Volts", "Amps",
    "effmot", "effprop", "adv", "CT", "CP", "DV", "eff",
    "Pelec", "Pprop", "cl_avg", "cd_avg",
]

RESULT_COLUMNS = [
    "config_id", "qprop_ok", "V", "rpm", "Dbeta", "T", "Q", "Pshaft",
    "CT", "CP", "FOM", "eff_aero", "cl_avg", "cd_avg",
]

print(f"Project folder        : {BASE_DIR}")
print(f"QPROP input folder    : {QPROP_INPUT_DIR}")
print(f"QPROP executable      : {QPROP_EXE_PATH}")
print(f"Motor file            : {MOTOR_FILE_SOURCE}")
print(f"Use short paths       : {USE_SHORT_RUNTIME_PATHS}")
print(f"Runtime prop folder   : {RUNTIME_PROP_DIR if USE_SHORT_RUNTIME_PATHS else QPROP_INPUT_DIR}")
print(f"Runtime output folder : {RUNTIME_OUTPUT_DIR if USE_SHORT_RUNTIME_PATHS else QPROP_PROJECT_OUTPUT_DIR}")
print(f"Output CSV            : {QPROP_RESULTS_CSV_PATH}")
print(f"Operating point       : V = {FREESTREAM_VELOCITY_M_S} m/s, rpm = {RPM}")
print(f"Workers               : {MAX_WORKERS}")


Project folder        : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset
QPROP input folder    : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_input
QPROP executable      : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop\qprop.exe
Motor file            : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\motor.mas
Use short paths       : True
Runtime prop folder   : C:\qp\props
Runtime output folder : C:\qp\out
Output CSV            : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\qprop_batch_results.csv
Operating point       : V = 0.0 m/s, rpm = 4000
Workers               : 16


## 3. Input validation

Before running an expensive batch, the notebook checks that the QPROP executable, the motor file, and the input folder exist.

Each propeller file must follow the naming convention:

```text
prop_0000.txt
prop_0001.txt
prop_0002.txt
...
```

The number inside the filename is used as `config_id`, so it remains aligned with the previous pipeline stages.


In [3]:

# Input validation

if not QPROP_INPUT_DIR.exists():
    raise FileNotFoundError(f"QPROP input folder not found: {QPROP_INPUT_DIR}")

if not QPROP_EXE_PATH.exists():
    raise FileNotFoundError(f"QPROP executable not found: {QPROP_EXE_PATH}")

if not MOTOR_FILE_SOURCE.exists():
    raise FileNotFoundError(f"motor.mas not found: {MOTOR_FILE_SOURCE}")

input_files = sorted(QPROP_INPUT_DIR.glob(f"*{QPROP_INPUT_EXTENSION}"))

if not input_files:
    raise FileNotFoundError(f"No QPROP input files found in {QPROP_INPUT_DIR}")

print(f"Input files found: {len(input_files)}")
print(f"First input file : {input_files[0].name}")
print(f"Last input file  : {input_files[-1].name}")


Input files found: 5000
First input file : prop_0000.txt
Last input file  : prop_4999.txt


## 4. Helper functions

The functions below keep the batch logic compact:

1. Extract `config_id` from each filename.
2. Prepare short runtime paths if required.
3. Run QPROP for one propeller file.
4. Parse the final valid numerical line from QPROP output.
5. Compute derived performance metrics.


In [4]:

# Helper functions

CONFIG_ID_PATTERN = re.compile(r"prop_(\d+)")


def extract_config_id(path):
    match = CONFIG_ID_PATTERN.search(path.stem)
    if match is None:
        raise ValueError(f"Could not extract config_id from filename: {path.name}")
    return int(match.group(1))


def output_path_for_prop(prop_file, output_dir):
    return output_dir / f"{prop_file.stem}_out.txt"


def prepare_runtime_files(source_input_files):
    """Prepare propeller input files and motor file for QPROP execution."""
    if USE_SHORT_RUNTIME_PATHS:
        RUNTIME_PROP_DIR.mkdir(parents=True, exist_ok=True)
        RUNTIME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        shutil.copy2(MOTOR_FILE_SOURCE, RUNTIME_MOTOR_FILE)

        runtime_files = []
        for src in source_input_files:
            dst = RUNTIME_PROP_DIR / src.name
            shutil.copy2(src, dst)
            runtime_files.append(dst)

        return runtime_files, RUNTIME_OUTPUT_DIR, RUNTIME_MOTOR_FILE

    QPROP_PROJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return source_input_files, QPROP_PROJECT_OUTPUT_DIR, MOTOR_FILE_SOURCE


def run_qprop(prop_file, output_dir, motor_file):
    """Run QPROP for a single propeller file and save stdout to a text file."""
    output_file = output_path_for_prop(prop_file, output_dir)
    stderr_file = output_dir / f"{prop_file.stem}_stderr.txt"

    if output_file.exists() and output_file.stat().st_size > 0 and not OVERWRITE_EXISTING_OUTPUTS:
        return output_file

    command = [
        str(QPROP_EXE_PATH),
        str(prop_file),
        str(motor_file),
        str(FREESTREAM_VELOCITY_M_S),
        str(RPM),
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            timeout=QPROP_TIMEOUT_S,
            check=False,
        )
        output_file.write_bytes(result.stdout)
        stderr_file.write_bytes(result.stderr)

        if result.returncode != 0 and output_file.stat().st_size == 0:
            output_file.write_text(f"QPROP_RETURN_CODE_{result.returncode}\n", encoding="utf-8")

    except subprocess.TimeoutExpired:
        output_file.write_text("TIMEOUT\n", encoding="utf-8")
        stderr_file.write_text("QPROP timed out.\n", encoding="utf-8")

    return output_file


def parse_qprop_output(output_file):
    """Parse the last valid numerical result line from a QPROP output file."""
    text = output_file.read_text(encoding="utf-8", errors="ignore")

    if not text.strip():
        return None

    if "TIMEOUT" in text:
        return None

    last_valid = None

    for raw_line in text.splitlines():
        line = raw_line.strip().lstrip("#").strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 13:
            continue

        try:
            values = [float(part) for part in parts[:len(QPROP_COLUMNS)]]
        except ValueError:
            continue

        # Skip common numeric header lines if present.
        if len(values) >= 3 and values[0] == 1.0 and values[1] == 2.0 and values[2] == 3.0:
            continue

        last_valid = dict(zip(QPROP_COLUMNS[:len(values)], values))

    return last_valid


def read_tip_radius_from_prop_file(prop_file):
    """Read R_tip from the second line of the generated QPROP input file."""
    lines = prop_file.read_text(encoding="utf-8", errors="ignore").splitlines()
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("!"):
            continue
        parts = stripped.split()
        # The first data line after the propeller name is:
        # Nblades R_tip(m) R_root(m)
        if len(parts) >= 3:
            try:
                int(float(parts[0]))
                return float(parts[1])
            except ValueError:
                continue
    raise ValueError(f"Could not read tip radius from {prop_file}")


def compute_fom(thrust_N, pshaft_W, radius_tip_m):
    if thrust_N is None or pshaft_W is None:
        return None
    if thrust_N <= 0 or pshaft_W <= 0 or radius_tip_m <= 0:
        return None

    disk_area_m2 = np.pi * radius_tip_m**2
    fom = thrust_N**1.5 / (np.sqrt(2.0 * RHO_AIR_KG_M3 * disk_area_m2) * pshaft_W)
    return float(fom)


def compute_aero_efficiency(CT, CP):
    if CT is None or CP is None:
        return None
    if CT <= 0 or CP <= 0:
        return None

    return float(CT / CP**(2.0 / 3.0))


def round_or_none(value, digits):
    if value is None or pd.isna(value):
        return None
    return round(float(value), digits)


## 5. Prepare runtime files

The source QPROP inputs remain in `./qprop_input`. If short runtime paths are enabled, temporary copies are created in a shorter folder before QPROP is executed.

This avoids the common issue where QPROP silently fails or reads the wrong files because the path is too long.


In [5]:

# Prepare runtime files

runtime_prop_files, runtime_output_dir, runtime_motor_file = prepare_runtime_files(input_files)

print(f"Runtime prop files : {len(runtime_prop_files)}")
print(f"Runtime prop dir   : {runtime_prop_files[0].parent}")
print(f"Runtime output dir : {runtime_output_dir}")
print(f"Runtime motor file : {runtime_motor_file}")


Runtime prop files : 5000
Runtime prop dir   : C:\qp\props
Runtime output dir : C:\qp\out
Runtime motor file : C:\qp\motor.mas


## 6. Run QPROP batch

Each input file is simulated independently. Existing output files are reused unless `OVERWRITE_EXISTING_OUTPUTS = True`.


In [6]:

# Run QPROP batch

files_to_run = []
for prop_file in runtime_prop_files:
    out_file = output_path_for_prop(prop_file, runtime_output_dir)
    needs_run = (
        OVERWRITE_EXISTING_OUTPUTS
        or not out_file.exists()
        or out_file.stat().st_size == 0
    )
    if needs_run:
        files_to_run.append(prop_file)

print(
    f"QPROP simulations: {len(files_to_run)} to run / {len(runtime_prop_files)} total "
    f"({len(runtime_prop_files) - len(files_to_run)} cached)"
)

start_time = time.time()

if files_to_run:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(run_qprop, prop_file, runtime_output_dir, runtime_motor_file): prop_file
            for prop_file in files_to_run
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="QPROP batch"):
            future.result()

elapsed_s = time.time() - start_time
print(f"QPROP batch completed in {elapsed_s:.1f} s")


QPROP simulations: 0 to run / 5000 total (5000 cached)
QPROP batch completed in 0.0 s


## 7. Parse QPROP outputs

For each configuration, the notebook reads the final valid numerical line from the QPROP output file and extracts the relevant metrics.

A failed or non-converged simulation is kept in the CSV with `qprop_ok = False`. This keeps the row count aligned with the input configurations.


In [7]:

# Parse QPROP outputs

records = []

for prop_file in tqdm(sorted(runtime_prop_files), desc="Parsing QPROP outputs"):
    config_id = extract_config_id(prop_file)
    output_file = output_path_for_prop(prop_file, runtime_output_dir)
    parsed = parse_qprop_output(output_file) if output_file.exists() else None

    try:
        radius_tip_m = read_tip_radius_from_prop_file(prop_file)
    except Exception:
        radius_tip_m = None

    if parsed is None or radius_tip_m is None:
        records.append({
            "config_id": config_id,
            "qprop_ok": False,
            "V": None,
            "rpm": None,
            "Dbeta": None,
            "T": None,
            "Q": None,
            "Pshaft": None,
            "CT": None,
            "CP": None,
            "FOM": None,
            "eff_aero": None,
            "cl_avg": None,
            "cd_avg": None,
        })
        continue

    thrust_N = parsed.get("T")
    pshaft_W = parsed.get("Pshaft")
    CT = parsed.get("CT")
    CP = parsed.get("CP")

    records.append({
        "config_id": config_id,
        "qprop_ok": True,
        "V": round_or_none(parsed.get("V"), 4),
        "rpm": round_or_none(parsed.get("rpm"), 2),
        "Dbeta": round_or_none(parsed.get("Dbeta"), 4),
        "T": round_or_none(thrust_N, 5),
        "Q": round_or_none(parsed.get("Q"), 6),
        "Pshaft": round_or_none(pshaft_W, 4),
        "CT": round_or_none(CT, 6),
        "CP": round_or_none(CP, 6),
        "FOM": round_or_none(compute_fom(thrust_N, pshaft_W, radius_tip_m), 5),
        "eff_aero": round_or_none(compute_aero_efficiency(CT, CP), 5),
        "cl_avg": round_or_none(parsed.get("cl_avg"), 5),
        "cd_avg": round_or_none(parsed.get("cd_avg"), 6),
    })

qprop_results_df = pd.DataFrame(records)
qprop_results_df = qprop_results_df.sort_values("config_id").reset_index(drop=True)
qprop_results_df = qprop_results_df[RESULT_COLUMNS]

print(f"Parsed rows : {len(qprop_results_df)}")
print(f"Converged   : {int(qprop_results_df['qprop_ok'].sum())}")
print(f"Failed      : {int((~qprop_results_df['qprop_ok']).sum())}")

qprop_results_df.head()


Parsing QPROP outputs: 100%|██████████| 5000/5000 [00:01<00:00, 2510.05it/s]

Parsed rows : 5000
Converged   : 4500
Failed      : 500


,config_id,qprop_ok,V,rpm,Dbeta,T,Q,Pshaft,CT,CP,FOM,eff_aero,cl_avg,cd_avg
0,0,True,0.0,4000.0,0.0,0.2571,0.002605,1.0910,0.03779,0.005715,0.64283,1.18222,0.2055,0.012130
1,1,True,0.0,4000.0,0.0,0.1757,0.001943,0.8138,0.04015,0.007400,0.54366,1.05730,0.3640,0.032330
2,2,True,0.0,4000.0,0.0,0.8104,0.011310,4.7370,0.05860,0.010220,0.69390,1.24432,0.4695,0.018540
3,3,True,0.0,4000.0,0.0,0.3085,0.003444,1.4430,0.04816,0.008145,0.64851,1.18967,0.2686,0.016780
4,4,True,0.0,4000.0,0.0,0.1543,0.001087,0.4553,0.02268,0.002385,0.71617,1.27053,0.2045,0.006952


## 8. Export results

The final CSV contains one row per input configuration and only the most important QPROP performance metrics.

Output file:

```text
./csv/qprop_batch_results.csv
```


In [8]:

# Export CSV

CSV_DIR.mkdir(parents=True, exist_ok=True)
qprop_results_df.to_csv(QPROP_RESULTS_CSV_PATH, index=False)

print(f"Saved QPROP results to: {QPROP_RESULTS_CSV_PATH}")
print(f"Rows                  : {len(qprop_results_df)}")
print(f"Columns               : {list(qprop_results_df.columns)}")


Saved QPROP results to: c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\qprop_batch_results.csv
Rows                  : 5000
Columns               : ['config_id', 'qprop_ok', 'V', 'rpm', 'Dbeta', 'T', 'Q', 'Pshaft', 'CT', 'CP', 'FOM', 'eff_aero', 'cl_avg', 'cd_avg']


## 9. Optional raw-output sync

When short runtime paths are used, raw QPROP text outputs are generated outside the project folder. This cell copies them back into `./qprop_output` for traceability.


In [9]:

# Optional raw-output sync

if SYNC_RAW_OUTPUTS_TO_PROJECT_DIR:
    QPROP_PROJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    copied = 0
    for output_file in runtime_output_dir.glob("*_out.txt"):
        shutil.copy2(output_file, QPROP_PROJECT_OUTPUT_DIR / output_file.name)
        copied += 1
    print(f"Copied {copied} raw QPROP output files to {QPROP_PROJECT_OUTPUT_DIR}")
else:
    print("Raw QPROP output sync disabled.")


Copied 5000 raw QPROP output files to c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\qprop_output


## 10. Final checks

These checks make sure that the batch output is structurally usable by the next pipeline stages.


In [10]:

# Final consistency checks

all_ok = True


def check(condition, message):
    global all_ok
    status = "OK" if bool(condition) else "FAIL"
    print(f"[{status:4s}] {message}")
    if not condition:
        all_ok = False


expected_config_ids = [extract_config_id(path) for path in sorted(runtime_prop_files)]
actual_config_ids = qprop_results_df["config_id"].astype(int).tolist()

check(QPROP_RESULTS_CSV_PATH.exists(), "output CSV exists")
check(list(qprop_results_df.columns) == RESULT_COLUMNS, "output columns match expected schema")
check(len(qprop_results_df) == len(runtime_prop_files), "one result row per QPROP input file")
check(actual_config_ids == expected_config_ids, "config_id values match input file names and remain sorted")
check(qprop_results_df["config_id"].is_unique, "config_id values are unique")
check(qprop_results_df["qprop_ok"].dtype == bool, "qprop_ok is boolean")

for col in ["T", "Q", "Pshaft", "CT", "CP", "FOM", "eff_aero", "cl_avg", "cd_avg"]:
    check(col in qprop_results_df.columns, f"{col} column exists")

readback_df = pd.read_csv(QPROP_RESULTS_CSV_PATH)
check(readback_df.shape == qprop_results_df.shape, "exported CSV can be read back with the same shape")

n_ok = int(qprop_results_df["qprop_ok"].sum())
n_failed = len(qprop_results_df) - n_ok
n_positive_thrust = int((qprop_results_df["T"] > 0).sum())
n_positive_power = int((qprop_results_df["Pshaft"] > 0).sum())

print()
print("QPROP result summary")
print("-" * 72)
print(f"Total configurations : {len(qprop_results_df)}")
print(f"Converged            : {n_ok}")
print(f"Failed               : {n_failed}")
print(f"Positive thrust      : {n_positive_thrust}")
print(f"Positive shaft power : {n_positive_power}")

print()
print("=" * 72)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED")
print("=" * 72)


[OK  ] output CSV exists
[OK  ] output columns match expected schema
[OK  ] one result row per QPROP input file
[OK  ] config_id values match input file names and remain sorted
[OK  ] config_id values are unique
[OK  ] qprop_ok is boolean
[OK  ] T column exists
[OK  ] Q column exists
[OK  ] Pshaft column exists
[OK  ] CT column exists
[OK  ] CP column exists
[OK  ] FOM column exists
[OK  ] eff_aero column exists
[OK  ] cl_avg column exists
[OK  ] cd_avg column exists
[OK  ] exported CSV can be read back with the same shape

QPROP result summary
------------------------------------------------------------------------
Total configurations : 5000
Converged            : 4500
Failed               : 500
Positive thrust      : 4292
Positive shaft power : 4497

ALL CHECKS PASSED
